In [ ]:
# Mount Google Drive to access input/output data
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
# Install required geospatial libraries
# rioxarray : raster I/O and spatial operations built on xarray + rasterio
# cartopy   : coordinate reference system and map projection support
!pip install rioxarray cartopy


In [ ]:
# ============================================================
# Imports and Configuration
# ============================================================
import os
from os import makedirs
from os.path import join
import numpy as np
import rioxarray
from glob import glob

# --- User configuration ---
# Folder containing the cloud-masked, land-masked LST GeoTIFFs
INPUT_DIR  = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/ECOSTRESS Tifs/Geolocated LST/cloudmasked_output/<YOUR_SUBFOLDER>"

# Folder where composite GeoTIFFs will be saved
OUTPUT_DIR = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/ECOSTRESS Tifs/Geolocated LST/cloudmasked_output/Composites"

makedirs(OUTPUT_DIR, exist_ok=True)

# Collect and sort all GeoTIFFs in the input folder
ST_masked_filenames = sorted(glob(join(INPUT_DIR, "*.tif")))


In [ ]:
# ============================================================
# Build and Save LST Composites
# ============================================================
# Computes four pixel-wise composites from the input LST stack:
#   - Maximum   : hottest observed temperature per pixel
#   - Median    : central tendency, robust to outliers
#   - Mean      : arithmetic average across valid observations
#   - Count     : number of valid (non-NaN) observations per pixel
#                 useful as a data-availability / confidence layer
#
# All composites ignore NaN values (cloud-masked or fill pixels).
# The first file in the sorted list is used as the spatial reference;
# all other rasters are reprojected to match it before stacking.
# Fill values (0) are converted to NaN before any reduction.
#
# Outputs are saved as tiled, LZW-compressed GeoTIFFs.
# The count composite is written as int16 (whole-number counts).
# ============================================================

def build_composite(filenames, output_dir, reduction, output_filename):
    """
    Load, align, and reduce a stack of LST GeoTIFFs into a single composite.

    Parameters
    ----------
    filenames       : list of str  - Sorted file paths to input GeoTIFFs.
    output_dir      : str          - Directory where the output GeoTIFF is saved.
    reduction       : str          - Composite type: 'max', 'median', 'mean', or 'count'.
    output_filename : str          - Basename for the output file (no path).

    Returns
    -------
    None. Writes a GeoTIFF to output_dir/output_filename.
    """
    # Open the first file as the spatial reference for reprojection
    ref_raster = rioxarray.open_rasterio(filenames[0]).squeeze('band', drop=True)

    # For the count composite, cast reference to int16 and set nodata=0
    if reduction == 'count':
        ref_raster = ref_raster.rio.write_nodata(0).astype(np.int16)

    # Load each raster, reproject to match the reference, and replace fill (0) with NaN
    aligned_arrays = []
    for f in filenames:
        with rioxarray.open_rasterio(f).squeeze('band', drop=True) as r:
            reproj = r.rio.reproject_match(ref_raster)
            arr = np.where(reproj.data == 0, np.nan, reproj.data)
            aligned_arrays.append(arr)

    # Stack to (n_files, rows, cols) and apply the chosen reduction
    stacked = np.stack(aligned_arrays, axis=0)

    if reduction == 'max':
        result = np.nanmax(stacked, axis=0)
    elif reduction == 'median':
        result = np.nanmedian(stacked, axis=0)
    elif reduction == 'mean':
        result = np.nanmean(stacked, axis=0)
    elif reduction == 'count':
        # Count valid (non-NaN) pixels per spatial position
        result = np.sum(~np.isnan(stacked), axis=0)
    else:
        raise ValueError(f"Unknown reduction '{reduction}'. Choose: max, median, mean, count.")

    # Write result into a copy of the reference DataArray to carry over CRS and transform
    composite = ref_raster.copy(deep=True)
    composite.data = result

    if reduction == 'count':
        composite = composite.astype(np.int16).rio.write_nodata(0)

    output_path = join(output_dir, output_filename)
    dtype = np.int16 if reduction == 'count' else None
    composite.rio.to_raster(
        output_path,
        tiled=True,
        compress='LZW',
        **({'dtype': dtype} if dtype else {})
    )


# --- Run all four composites ---
composites = [
    ('max',    'ECOSTRESS_LST_Max_Composite.tif'),
    ('median', 'ECOSTRESS_LST_Median_Composite.tif'),
    ('mean',   'ECOSTRESS_LST_Mean_Composite.tif'),
    ('count',  'ECOSTRESS_LST_PixelCount_Composite.tif'),
]

for reduction, filename in composites:
    build_composite(ST_masked_filenames, OUTPUT_DIR, reduction, filename)
